In [7]:
# Install dependencies (runs once per Colab session)
!pip install -q openai langchain-groq python-dotenv ipywidgets

In [8]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from langchain_groq import ChatGroq
# Load environment variables
load_dotenv(find_dotenv())
# Load API key from Google Colab Secrets
if not os.environ.get("GROQ_API_KEY"):
    try:
      from google.colab import userdata
      os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
      print("Successfully retrieved GROQ_API_KEY from Google Colab Secrets!")
    except Exception:
      pass
api_key = os.environ.get("GROQ_API_KEY","").strip()

if not api_key:
    raise ValueError("GROQ_API_KEY is missing! Please configure it in Colab Secrets.")

# Model selection
MODEL_NAME = "openai/gpt-oss-120b"
client = OpenAI(
      api_key=api_key,
      base_url="https://api.groq.com/openai/v1"
)
# Initialize LangChain ChatGroq
llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    groq_api_key=api_key
)
print(f"Groq client connected successfully! Target Model: {MODEL_NAME}")

Successfully retrieved GROQ_API_KEY from Google Colab Secrets!
Groq client connected successfully! Target Model: openai/gpt-oss-120b


In [9]:
messages = [
      { "role": "system","content": "Role: Shop assistant. Facts: We sell stationery. Rules: Be polite. Format: Reply briefly."},
      {"role": "user","content": "Do you sell notebooks?"}
]

response = client.chat.completions.create(
  model=MODEL_NAME,
  messages=messages,
  temperature=0.2
)
new_message = "The cake was damaged"

messages = [
    {"role": "system", "content": "Classify messages as order, question or complaint."},
    {"role": "user", "content": "I need 2 plum cakes"},
    {"role": "assistant", "content": "order"},
    {"role": "user", "content": "Do you have eggless cakes?"},
    {"role": "assistant", "content": "question"},
    {"role": "user", "content": "My cake was damaged"},
    {"role": "assistant", "content": "complaint"},
    {"role": "user", "content": new_message}
]
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    temperature=0.2
)
print(response.choices[0].message.content)

complaint


In [10]:
import json

messages = [
    {"role": "system", "content": 'Extract the order. Return ONLY JSON with keys: name, item, qty, date, eggless. Use a number for qty and true/false for eggless.'},
    {"role": "user", "content": "My name is Purnima. I want 2 plum cakes for Friday, eggless."}
]

response = client.chat.completions.create(
  model=MODEL_NAME,
  messages=messages,
  temperature=0.2
)
reply = response.choices[0].message.content
try:
   order = json.loads(reply)
   print(order)
   print("Item:", order["item"])
   print("Quantity:", order["qty"])
except json.JSONDecodeError:
    print("The response was not valid JSON:")
    print(reply)

{'name': 'Purnima', 'item': 'plum cake', 'qty': 2, 'date': 'Friday', 'eggless': True}
Item: plum cake
Quantity: 2


In [11]:
import json

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}
SYSTEM_PROMPT = f"""
ROLE:
You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
- Menu: {json.dumps(MENU)}
- Orders require 24 hours' notice.
- Eggless versions are available for all cakes.

RULES:
- Never promise a delivery time.
- Never invent products or prices.
- If unsure, say you will check with the owner.

FORMAT:
- Be polite and clear.
- Keep replies under 50 words.
- Reply in English or Telugu according to the customer.
"""
print(SYSTEM_PROMPT)


ROLE:
You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
- Menu: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
- Orders require 24 hours' notice.
- Eggless versions are available for all cakes.

RULES:
- Never promise a delivery time.
- Never invent products or prices.
- If unsure, say you will check with the owner.

FORMAT:
- Be polite and clear.
- Keep replies under 50 words.
- Reply in English or Telugu according to the customer.



In [12]:

messages = [
    ("system", """
    You are a customer message classifier for Sweet Crumbs Home Bakery.
    Classify each message into one category:
    order, inquiry, or complaint.
    Reply with only the category name.
    """),
    ("human", "I want to order two chocolate cakes."),
    ("ai", "order"),
    ("human", "What is the price of a cupcake?"),
    ("ai", "inquiry"),
    ("human", "My cake arrived damaged."),
    ("ai", "complaint"),
    ("human", "Can I order one plum cake for tomorrow?")
]
response = llm.invoke(messages)

print("Customer message category:", response.content)


Customer message category: order


In [13]:

history = []

def chat(user_message):
    history.append(("human", user_message))
    messages = [("system", SYSTEM_PROMPT)] + history
    response = llm.invoke(messages)
    history.append(("ai", response.content))
    return response.content
print(chat("Hi, my name is Purnima."))
print(chat("What is my name?"))


Hello Purnima! Welcome to Sweet Crumbs Home Bakery. How can I assist you today—perhaps with our plum cake, chocolate cake, or cupcakes? Let me know your preferences!
Your name is Purnima.


In [15]:
MAX_TURNS = 2

def chat(user_message):
  history.append(("human", user_message))
  history[:] = history[-(MAX_TURNS * 2):]
  messages = [("system", SYSTEM_PROMPT)] + history
  response = llm.invoke(messages)
  history.append(("ai", response.content))
  return response.content
print(chat("What cakes are available?"))


We offer:

- Plum cake – ₹500 (eggless available)  
- Chocolate cake – ₹650 (eggless available)  
- Cupcake – ₹80 each (eggless available)

All orders need 24 hours’ notice. Let us know what you’d like!


In [16]:
def calculate_price(item: str, quantity: int) -> int:
    return MENU[item.lower()] * quantity
print("Total price:", calculate_price("plum cake", 2))

Total price: 1000


In [17]:
def log_call(func):
  def wrapper(*args):
    print("Function started:", func.__name__)
    result = func(*args)
    print("Function completed!")
    return result
  return wrapper
@log_call
def calculate_price(item: str, quantity: int) -> int:
    return MENU[item.lower()] * quantity
print("Total:", calculate_price("plum cake", 2))

Function started: calculate_price
Function completed!
Total: 1000


In [18]:
try:
    print("Total:", calculate_price("plum cake", 2))
    print("Total:", calculate_price("unknown cake", 1))
except KeyError:
     print("Sorry, this item is not available.")

Function started: calculate_price
Function completed!
Total: 1000
Function started: calculate_price
Sorry, this item is not available.
